# Thesis — MLRan ransomware detection

This notebook runs the thesis feature-selection pipeline on the **MLRan** behavioural dataset
(4,880 samples: 2,550 goodware + 2,330 ransomware from 64 families).

**Pipeline**
1. Load the MI-selected set (24,162 features; official temporal split 3,905 train / 975 test).
2. Rank features with the vote rule (`paper1_votes`: MI, F, RFE, |corr|).
3. Evaluate on the temporal split: vote-selected set, top-483, the paper's RFE-483 set, all MI features.
4. Evaluate under a family-disjoint `GroupKFold` protocol.
5. Inspect feature names, overlap with the paper's RFE set, and stability across folds.

The MI CSVs are downloaded automatically from Zenodo (record 23403681) on first run.
Runtime is roughly 10–15 minutes on a laptop; the family-disjoint section is the slowest.

The previous RADAR-based version of this notebook is kept as `Thesis_Final_original_backup.ipynb`.


## 1. Setup

Loads the libraries, resolves the data folder (Google Drive on Colab, the notebook folder
locally), and downloads the MI set from Zenodo if it is missing.


In [ ]:
import json
import os
import time
import urllib.request
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# --- data folder -----------------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Thesis"
except Exception:
    DATA_DIR = os.getcwd()

SUPP = os.path.join(DATA_DIR, "MLRan_supplementary")
os.makedirs(SUPP, exist_ok=True)
print("data folder:", SUPP)

# --- download the MI set if missing (Zenodo record 23403681) ---------------
ZENODO = "https://zenodo.org/api/records/23403681"
NEEDED = [
    "MLRan_X_train_MI_01.csv",
    "MLRan_X_test_MI_01.csv",
    "MI_selected_feature_names_dic.json",
    "MLRan_labels.csv",
    "MLRan_X_train_RFE.csv",
    "MLRan_X_test_RFE.csv",
    "RFE_selected_feature_names_dic.json",
    "mlran_dataset_metadata.csv",
]

missing = [f for f in NEEDED if not os.path.exists(os.path.join(SUPP, f))]
if missing:
    with urllib.request.urlopen(ZENODO) as r:
        record = json.load(r)
    links = {f["key"]: f["links"]["self"] for f in record["files"]}
    for f in missing:
        print("downloading", f, flush=True)
        urllib.request.urlretrieve(links[f], os.path.join(SUPP, f))
print("files ready:", len(NEEDED), "required")


## 2. Load the MLRan MI set

The MI CSVs already contain `sample_id`, `sample_type` (0 = goodware, 1 = ransomware),
`family_label` and `type_label`, so no label join is needed. Features are read as
`float32` to keep the 24k-column table in memory.


In [ ]:
META = ["sample_id", "sample_type", "family_label", "type_label"]


def read_wide(path):
    """Read a wide MLRan CSV with float32 features (memory-friendly)."""
    head = pd.read_csv(path, nrows=0)
    cols = list(head.columns)
    meta = [c for c in cols if c in META]
    feats = [c for c in cols if c not in meta]
    dtypes = {c: np.float32 for c in feats}
    dtypes.update({c: np.int64 for c in meta})
    return pd.read_csv(path, dtype=dtypes)


t0 = time.time()
tr = read_wide(os.path.join(SUPP, "MLRan_X_train_MI_01.csv"))
te = read_wide(os.path.join(SUPP, "MLRan_X_test_MI_01.csv"))
print(f"loaded in {time.time() - t0:.1f}s")

feats = [c for c in tr.columns if c not in META]
assert feats == [c for c in te.columns if c not in META]

Xtr = tr[feats].to_numpy(dtype=np.float32)
Xte = te[feats].to_numpy(dtype=np.float32)
ytr = tr["sample_type"].to_numpy(dtype=np.int8)
yte = te["sample_type"].to_numpy(dtype=np.int8)
ids_tr = tr["sample_id"].to_numpy()
ids_te = te["sample_id"].to_numpy()
del tr, te

print("train:", Xtr.shape, "| test:", Xte.shape)
print("train classes:", np.bincount(ytr), "| test classes:", np.bincount(yte))
print("train/test sample_id overlap:", len(set(ids_tr) & set(ids_te)))
print("feature IDs:", feats[0], "...", feats[-1], "| n =", len(feats))


## 3. Vote-based feature selection

`paper1_votes` combines four rankings: mutual information, ANOVA F, RFE, and |correlation|.
A feature is kept when at least two of the four rank it above their mean.

Two changes make the rule tractable on 24k features: MI is computed with
`discrete_features=True` (the MLRan features are integer counts), and RFE runs on the
top 2,000 MI features rather than all 24k. The top-483 set is the size-matched comparison
with the paper's 483-feature RFE set.


In [ ]:
from sklearn.feature_selection import RFE, f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression


def paper1_votes(X, y, rfe_top=2000, rfe_k=10, rfe_step=20, random_state=42):
    """Vote-based selection: MI, F, RFE, |corr|.

    MI is computed on integer count features (discrete_features=True).
    RFE runs on the top-rfe_top MI features so the rule stays tractable
    on the 24k-feature MLRan set.
    """
    n = X.shape[1]
    mi = mutual_info_classif(X, y, discrete_features=True, random_state=random_state)
    F = np.nan_to_num(f_classif(X, y)[0])
    keep = np.zeros(n, dtype=bool)
    if len(np.unique(y)) > 1:
        top = np.argsort(-mi)[: min(rfe_top, n)]
        rfe = RFE(
            LogisticRegression(max_iter=2000),
            n_features_to_select=min(rfe_k, len(top)),
            step=max(1, min(rfe_step, len(top) // 20)),
        )
        rfe.fit(X[:, top], y)
        keep[top[rfe.support_]] = True
    corr = np.nan_to_num(np.array([abs(np.corrcoef(X[:, j], y)[0, 1]) for j in range(n)]))
    votes = (
        (mi > mi.mean()).astype(int)
        + (F > F.mean()).astype(int)
        + keep.astype(int)
        + (corr > corr.mean()).astype(int)
    )
    return votes, mi


t0 = time.time()
votes, mi = paper1_votes(Xtr, ytr)
order = np.lexsort((-mi, -votes))  # votes descending, then MI descending
sel_vote = np.where(votes >= 2)[0]
sel_top = order[:483]
print(f"selection in {time.time() - t0:.1f}s")
print("vote histogram:", np.bincount(votes, minlength=5))
print(f"votes>=2: {len(sel_vote)} features | top-483: {len(sel_top)}")


## 4. Temporal split evaluation

The official split is temporal: older samples in train, newer in test. The selection is
computed on the training split only. Four feature sets are compared: the vote-selected set,
the top-483 set, the paper's RFE-483 set, and all 24,162 MI features.


In [ ]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score, roc_auc_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False


def make_models():
    models = {
        "LR": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
        "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5, n_jobs=-1)),
        "NB": GaussianNB(),
        "DT": DecisionTreeClassifier(random_state=42),
        "RF": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
        "AB": AdaBoostClassifier(n_estimators=50, random_state=42),
    }
    if HAS_XGB:
        models["XGB"] = XGBClassifier(
            n_estimators=100, max_depth=6, n_jobs=-1, random_state=42, eval_metric="logloss"
        )
    return models


def evaluate(Xa, ya, Xb, yb, idx, model_name):
    model = make_models()[model_name]
    model.fit(Xa[:, idx], ya)
    pred = model.predict(Xb[:, idx])
    proba = model.predict_proba(Xb[:, idx])[:, 1] if hasattr(model, "predict_proba") else None
    return {
        "bal_acc": balanced_accuracy_score(yb, pred),
        "f1": f1_score(yb, pred, zero_division=0),
        "fnr": 1 - recall_score(yb, pred, zero_division=0),
        "auc": roc_auc_score(yb, proba) if proba is not None else np.nan,
    }


# --- the paper's RFE-483 set ---
rfe_tr = read_wide(os.path.join(SUPP, "MLRan_X_train_RFE.csv"))
rfe_te = read_wide(os.path.join(SUPP, "MLRan_X_test_RFE.csv"))
rfe_feats = [c for c in rfe_tr.columns if c not in META]
Xrfe_tr = rfe_tr[rfe_feats].to_numpy(dtype=np.float32)
Xrfe_te = rfe_te[rfe_feats].to_numpy(dtype=np.float32)
yrfe_tr = rfe_tr["sample_type"].to_numpy(dtype=np.int8)
yrfe_te = rfe_te["sample_type"].to_numpy(dtype=np.int8)
del rfe_tr, rfe_te

sets = {
    "vote>=2": (Xtr, ytr, Xte, yte, sel_vote),
    "top-483": (Xtr, ytr, Xte, yte, sel_top),
    "RFE-483": (Xrfe_tr, yrfe_tr, Xrfe_te, yrfe_te, np.arange(len(rfe_feats))),
    "all-MI": (Xtr, ytr, Xte, yte, np.arange(Xtr.shape[1])),
}

rows = []
for set_name, (Xa, ya, Xb, yb, idx) in sets.items():
    for model_name in make_models():
        t0 = time.time()
        r = evaluate(Xa, ya, Xb, yb, idx, model_name)
        rows.append({"set": set_name, "model": model_name, **r, "sec": time.time() - t0})
        print(f"{set_name:8s} {model_name:4s} bal_acc={r['bal_acc']:.3f} ({time.time() - t0:.0f}s)")

temporal = pd.DataFrame(rows)
print("\nbalanced accuracy by set x model:")
print(temporal.pivot_table(index="model", columns="set", values="bal_acc").round(3))
temporal.to_csv(os.path.join(SUPP, "results_temporal.csv"), index=False)
print("saved:", os.path.join(SUPP, "results_temporal.csv"))


## 5. Feature-count sensitivity

How many features are enough? The vote order is truncated at 25, 50, 100, 200, 483 and
1,000 features and evaluated on the temporal test split.


In [ ]:
ks = [25, 50, 100, 200, 483, 1000]
sweep_rows = []
for k in ks:
    idx = order[:k]
    for model_name in ["LR", "RF", "XGB"] if HAS_XGB else ["LR", "RF"]:
        r = evaluate(Xtr, ytr, Xte, yte, idx, model_name)
        sweep_rows.append({"k": k, "model": model_name, **r})

sweep = pd.DataFrame(sweep_rows)
print(sweep.pivot_table(index="k", columns="model", values="bal_acc").round(3))


## 6. Family-disjoint protocol

The family-disjoint protocol asks whether the pipeline generalises to unseen ransomware
families. `GroupKFold` holds out whole families; goodware is grouped by submission year so
each fold still contains benign samples. Selection is recomputed inside every fold, so no
information leaks from the held-out families.


In [ ]:
from sklearn.model_selection import GroupKFold

# family-disjoint groups: ransomware by family, goodware by submission year
meta = pd.read_csv(
    os.path.join(SUPP, "mlran_dataset_metadata.csv"),
    usecols=["sample_id", "ransomware_family", "first_submission_date_year"],
)
meta = meta.drop_duplicates("sample_id").set_index("sample_id")
year = meta["first_submission_date_year"].fillna(0).astype(int)
g = np.where(
    meta["ransomware_family"] == "goodware",
    "goodware_" + year.astype(str),
    meta["ransomware_family"],
)
gser = pd.Series(g, index=meta.index)

# combined MI set (train + test) for the family-disjoint protocol
Xall = np.vstack([Xtr, Xte])
yall = np.concatenate([ytr, yte])
ids_all = np.concatenate([ids_tr, ids_te])
gall = gser.reindex(ids_all).fillna("unknown").to_numpy()

print("groups:", len(np.unique(gall)), "| class balance:", np.bincount(yall))

gkf = GroupKFold(n_splits=5)
fam_rows = []
fold_sets = []
for fold, (itr, ite) in enumerate(gkf.split(Xall, yall, gall)):
    v, m = paper1_votes(Xall[itr], yall[itr])
    o = np.lexsort((-m, -v))
    idx = o[:483]
    fold_sets.append(set(idx))
    for model_name in make_models():
        r = evaluate(Xall[itr], yall[itr], Xall[ite], yall[ite], idx, model_name)
        fam_rows.append({"fold": fold, "model": model_name, **r})
    print(f"fold {fold}: train {len(itr)} test {len(ite)} | test classes {np.bincount(yall[ite])}")

fam = pd.DataFrame(fam_rows)
print(fam.groupby("model")[["bal_acc", "f1", "fnr", "auc"]].agg(["mean", "std"]).round(3))
fam.to_csv(os.path.join(SUPP, "results_family_disjoint.csv"), index=False)
print("saved:", os.path.join(SUPP, "results_family_disjoint.csv"))


## 7. Feature analysis

Maps the top-ranked feature IDs to their behavioural names, measures overlap with the
paper's RFE set, and checks how stable the top-483 set is across family folds.


In [ ]:
with open(os.path.join(SUPP, "MI_selected_feature_names_dic.json")) as f:
    mi_names = json.load(f)
with open(os.path.join(SUPP, "RFE_selected_feature_names_dic.json")) as f:
    rfe_names = json.load(f)

top = order[:20]
tbl = pd.DataFrame(
    {
        "feature_id": [feats[j] for j in top],
        "name": [mi_names.get(feats[j], "")[:90] for j in top],
        "votes": [int(votes[j]) for j in top],
        "MI": [round(float(mi[j]), 4) for j in top],
    }
)
print(tbl.to_string(index=False))

sel_ids = {feats[j] for j in sel_top}
rfe_ids = set(rfe_feats)
print(f"\ntop-483 overlap with RFE-483: {len(sel_ids & rfe_ids)} / 483")
print(f"RFE-483 features present in the MI set: {len(rfe_ids & set(feats))} / 483")
overlap = sorted(sel_ids & rfe_ids)
print("overlap examples:", [rfe_names.get(str(f), "")[:70] for f in overlap[:8]])

# modality breakdown of the selected sets
def modality(fid):
    nm = mi_names.get(str(fid), "")
    return nm.split(":")[0] if ":" in nm else "OTHER"


mod = pd.DataFrame(
    {
        "vote>=2": pd.Series([modality(feats[j]) for j in sel_vote]).value_counts(),
        "top-483": pd.Series([modality(feats[j]) for j in sel_top]).value_counts(),
    }
).fillna(0).astype(int)
print("\nmodality breakdown:")
print(mod.to_string())

# export the full ranking for the thesis appendix
export = pd.DataFrame(
    {
        "feature_id": [feats[j] for j in order],
        "name": [mi_names.get(str(feats[j]), "") for j in order],
        "votes": votes[order],
        "MI": mi[order].round(6),
        "selected_vote": np.isin(order, sel_vote),
        "selected_top483": np.isin(order, sel_top),
    }
)
export.to_csv(os.path.join(SUPP, "selected_features.csv"), index=False)
print("\nsaved:", os.path.join(SUPP, "selected_features.csv"))

jacs = []
for i in range(len(fold_sets)):
    for j in range(i + 1, len(fold_sets)):
        a, b = fold_sets[i], fold_sets[j]
        jacs.append(len(a & b) / len(a | b))
print("mean Jaccard across family folds:", round(float(np.mean(jacs)), 3))


## 8. Notes

- The MI set is already feature-selected; the vote rule re-ranks it. The top-483 set is a
  size-matched comparison with the paper's RFE-483 set, not a reimplementation of RFE.
- The temporal split is the paper's protocol; the family-disjoint `GroupKFold` is a second,
  harder protocol. Report both rather than pooling them.
- MI is computed with `discrete_features=True` on integer counts. If you change the feature
  representation (e.g. scaled floats), switch MI back to continuous.
- The family-disjoint section recomputes selection in every fold (~75 s per fold on 24k
  features), which is why it is the slowest part of the notebook.
